# MODULE 3 : Développement Applicatif
## III : Programmation Orientée Objet (POO)



### Objectifs du jour
- Comprendre pourquoi la POO structure mieux un projet réel
- Maîtriser classes, attributs, méthodes, encapsulation
- Gérer les relations entre objets (composition)
- Réécrire le projet fil rouge en version orientée objet

### Déroulé de la journée
1. Pourquoi la POO ? 
2. Classes, attributs, méthodes
3. Encapsulation et propriétés (`@property`) 
4. Relations entre objets (composition) 
5. Atelier : le projet fil rouge en POO

---
## 1. Pourquoi la POO ? 

Relisez votre code du Jour 2 : `clients`, `commandes`, `prochain_id_client` étaient des **variables globales**. Tout le monde peut les modifier n'importe comment, sans passer par vos fonctions de validation. C'est risqué dans une vraie application.

**La POO répond à 3 problèmes concrets :**

| Problème | Solution POO |
|---|---|
| Variables globales modifiables par tous | **Encapsulation** : les données sont protégées dans l'objet |
| Fonctions et données séparées, difficile de savoir quelle fonction va avec quelles données | **Regroupement** : données + comportements dans une même classe |
| Duplication de logique similaire pour Client/Commande/Fournisseur | **Héritage/réutilisation** (aperçu rapide aujourd'hui) |


---
## 2. Classes, attributs, méthodes - rappel

In [2]:
import re

def valider_email(email):
    motif = r"^[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return re.match(motif, email) is not None

def valider_telephone(telephone):
    return telephone.isdigit() and len(telephone) == 10


class Client:
    """Représente un client de l'entreprise et ses commandes."""

    def __init__(self, id_client, nom, email, telephone):
        if not nom or not nom.strip():
            raise ValueError("Le nom du client est obligatoire.")
        if not valider_email(email):
            raise ValueError(f"Email invalide : {email}")
        if not valider_telephone(telephone):
            raise ValueError(f"Téléphone invalide : {telephone}")

        self.id_client = id_client
        self.nom = nom.strip()
        self.email = email
        self.telephone = telephone
        self.commandes = []  

    def ajouter_commande(self, commande):
        """Ajoute une commande à ce client."""
        self.commandes.append(commande)

    def total_depense(self):
        """Calcule le total dépensé par ce client."""
        return round(sum(c.total() for c in self.commandes), 2)

    def __repr__(self):
        return f"Client(id={self.id_client}, nom='{self.nom}')"


# Remarquez : la méthode __init__ VALIDE les données à la création.
# Il devient IMPOSSIBLE de créer un Client invalide -> c'est ça, l'encapsulation en action.

try:
    client_invalide = Client(1, "Test", "pas-un-email", "123")
except ValueError as erreur:
    print(f"Création bloquée comme attendu : {erreur}")

Création bloquée comme attendu : Email invalide : pas-un-email


In [8]:
class Commande:
    """Représente une commande passée par un client."""

    def __init__(self, id_commande, client, date, produit, quantite, prix_unitaire):
        if quantite <= 0:
            raise ValueError("La quantité doit être strictement positive.")
        if prix_unitaire < 0:
            raise ValueError("Le prix ne peut pas être négatif.")

        self.id_commande = id_commande
        self.client = client          # relation : une commande "connaît" son client
        self.date = date
        self.produit = produit
        self.quantite = quantite
        self.prix_unitaire = prix_unitaire

    def total(self):
        """Calcule le montant total de cette commande."""
        return round(self.quantite * self.prix_unitaire, 2)

    def appliquer_remise(self, pourcentage):
        """Applique une remise en pourcentage sur le prix unitaire."""
        if not (0 <= pourcentage <= 100):
            raise ValueError("Le pourcentage doit être entre 0 et 100.")
        self.prix_unitaire = round(self.prix_unitaire * (1 - pourcentage / 100), 2)

    def __repr__(self):
        return f"Commande(id={self.id_commande}, produit='{self.produit}', total={self.total()}€)"


# Test de la relation Client <-> Commande
client1 = Client(1, "Amina Traoré", "amina@example.com", "0612345678")
commande1 = Commande(1, client1, "2026-08-20", "Clavier mécanique", 2, 45.0)
client1.ajouter_commande(commande1)

type(commande1)

__main__.Commande

In [ ]:
print(client1)
print(client1.commandes)
print("Total dépensé :", client1.total_depense(), "€")

Client(id=1, nom='Amina Traoré')
[Commande(id=1, produit='Clavier mécanique', total=90.0€)]
Total dépensé : 90.0 €


---
## 3. Encapsulation avancée avec `@property`

Actuellement, rien n'empêche un développeur (vous, un collègue) de faire `client1.email = "n_importe_quoi"` après coup, en contournant la validation. En Python, on protège ça avec des attributs "privés" (convention `_nom`) et des **propriétés**.

In [9]:
class ClientSecurise:
    """Version avec encapsulation stricte : l'email est toujours validé, même après modification."""

    def __init__(self, id_client, nom, email, telephone):
        self.id_client = id_client
        self.nom = nom
        self.email = email  # passe automatiquement par le setter ci-dessous
        self.telephone = telephone
        self.commandes = []

    @property
    def email(self):
        """Getter : lecture de l'email."""
        return self._email

    @email.setter
    def email(self, valeur):
        """Setter : toute modification de l'email est validée, à la création ET après coup."""
        if not valider_email(valeur):
            raise ValueError(f"Email invalide : {valeur}")
        self._email = valeur

    def __repr__(self):
        return f"ClientSecurise(id={self.id_client}, nom='{self.nom}')"


client_secu = ClientSecurise(1, "Amina Traoré", "amina@example.com", "0612345678")
print("Email initial :", client_secu.email)

client_secu.email = "nouveau.email@example.com"  # modification valide -> OK
print("Email modifié :", client_secu.email)

try:
    client_secu.email = "@gmail"  # tentative de modification invalide -> bloquée
except ValueError as erreur:
    print(f"Modification bloquée comme attendu : {erreur}")

Email initial : amina@example.com
Email modifié : nouveau.email@example.com
Modification bloquée comme attendu : Email invalide : @gmail


---
## 4. Aperçu : héritage

Imaginons que l'entreprise ait aussi des **clients VIP** qui bénéficient d'une remise automatique. Plutôt que dupliquer le code, on **hérite** de `Client`.

In [3]:
class ClientVIP(Client):
    """Client bénéficiant d'une remise automatique sur toutes ses commandes."""

    def __init__(self, id_client, nom, email, telephone, pourcentage_remise=10):
        super().__init__(id_client, nom, email, telephone)  # réutilise le constructeur parent
        self.pourcentage_remise = pourcentage_remise

    def total_depense(self):
        """Redéfinit le calcul pour appliquer la remise VIP (polymorphisme)."""
        total_brut = super().total_depense()
        return round(total_brut * (1 - self.pourcentage_remise / 100), 2)


client_vip = ClientVIP(2, "Jean Kouassi", "jean.k@example.com", "0623456789", pourcentage_remise=15)
commande_vip = Commande(2, client_vip, "2026-08-21", "Écran 24 pouces", 1, 150.0)
client_vip.ajouter_commande(commande_vip)

print("Total brut de la commande :", commande_vip.total(), "€")
print("Total dépensé (avec remise VIP) :", client_vip.total_depense(), "€")

# C'est ça, le polymorphisme : Client.total_depense() et ClientVIP.total_depense()
# portent le même nom mais un comportement différent selon le type d'objet.

Total brut de la commande : 150.0 €
Total dépensé (avec remise VIP) : 127.5 €


---
## 5. Le projet fil rouge en version POO complète

On assemble maintenant tout dans une classe `GestionCommerciale` qui va gérer l'ensemble des clients, c'est cette classe qui servira de "moteur" pour l'interface graphique.

In [4]:
class GestionCommerciale:
    """Gère l'ensemble des clients et centralise les opérations métier."""

    def __init__(self):
        self._clients = {}          # {id_client: Client}
        self._prochain_id_client = 1
        self._prochain_id_commande = 1

    def ajouter_client(self, nom, email, telephone):
        """Crée et enregistre un nouveau client. Retourne l'objet Client créé."""
        for client in self._clients.values():
            if client.email == email:
                raise ValueError(f"Un client avec l'email {email} existe déjà.")

        client = Client(self._prochain_id_client, nom, email, telephone)
        self._clients[client.id_client] = client
        self._prochain_id_client += 1
        return client

    def ajouter_commande(self, id_client, date, produit, quantite, prix_unitaire):
        """Crée une commande pour un client existant. Retourne l'objet Commande créé."""
        if id_client not in self._clients:
            raise ValueError(f"Client inconnu (id={id_client}).")

        client = self._clients[id_client]
        commande = Commande(self._prochain_id_commande, client, date, produit, quantite, prix_unitaire)
        client.ajouter_commande(commande)
        self._prochain_id_commande += 1
        return commande

    def lister_clients(self):
        """Retourne la liste de tous les clients."""
        return list(self._clients.values())

    def obtenir_client(self, id_client):
        """Retourne un client par son id, ou None s'il n'existe pas."""
        return self._clients.get(id_client)

    def meilleur_client(self):
        """Retourne le client ayant le plus dépensé, ou None s'il n'y a aucun client."""
        if not self._clients:
            return None
        return max(self._clients.values(), key=lambda c: c.total_depense())


# Démonstration complète du moteur métier
gestion = GestionCommerciale()

c1 = gestion.ajouter_client("Amina Traoré", "amina@example.com", "0612345678")
c2 = gestion.ajouter_client("Jean Kouassi", "jean.k@example.com", "0623456789")

gestion.ajouter_commande(c1.id_client, "2026-08-20", "Clavier mécanique", 2, 45.0)
gestion.ajouter_commande(c1.id_client, "2026-08-22", "Souris sans fil", 1, 25.0)
gestion.ajouter_commande(c2.id_client, "2026-08-21", "Écran 24 pouces", 1, 150.0)

print("Tous les clients :", gestion.lister_clients())
print("\nMeilleur client :", gestion.meilleur_client())
print("Son total dépensé :", gestion.meilleur_client().total_depense(), "€")

Tous les clients : [Client(id=1, nom='Amina Traoré'), Client(id=2, nom='Jean Kouassi')]

Meilleur client : Client(id=2, nom='Jean Kouassi')
Son total dépensé : 150.0 €
